# Unsupervised LoRaWAN reception profiles
**Objective:** construct an interpretable profile, fit PCA and k-means on held-in addresses, and compare reconstruction with an autoencoder.
**Data:** public LoED sample, CC BY 4.0. No incident labels are supplied by this exercise.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


## 1. Build one profile visibly from reception rows
A profile is one UTC date, gateway and observed address. It is not verified physical-device identity. We keep CRC-valid data uplinks and at least 20 receptions per profile.

In [ ]:
paths=sorted((ROOT/'data/raw/loed').glob('*.csv'))
raw=pd.concat([pd.read_csv(path,dtype={'device_address':str,'gateway':str,'mtype':str}) for path in paths],ignore_index=True)
raw=raw[raw.crc_status.eq(1)&raw.mtype.isin(['010','100'])&raw.device_address.notna()].copy()
raw['time']=pd.to_datetime(raw.time,utc=True);raw['day']=raw.time.dt.strftime('%Y-%m-%d');raw=raw.sort_values('time')
key,g=next((key,g) for key,g in raw.groupby(['day','gateway','device_address'],sort=True) if len(g)>=20)
gaps=g.time.diff().dt.total_seconds().dropna().to_numpy()
profile={'log_count':np.log1p(len(g)),'rssi_mean':g.rssi.mean(),'rssi_std':g.rssi.std(ddof=0),'snr_mean':g.snr.mean(),'snr_std':g.snr.std(ddof=0),'sf_mean':g.spreading_factor.mean(),'sf_std':g.spreading_factor.std(ddof=0),'log_median_gap':np.log1p(np.median(gaps)),'log_gap_iqr':np.log1p(np.quantile(gaps,.75)-np.quantile(gaps,.25)),'frequency_count':g.frequency.nunique()}
print('Profile key:',key);display(pd.Series(profile));del raw


## 2. Separate observed addresses before fitting transformations

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from prepare_loed import FEATURES

d=pd.read_csv(ROOT/'data/processed/loed_profiles.csv',dtype={'address':str,'gateway':str});X=d[FEATURES].to_numpy()
tv,te=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,groups=d.address))
t,v=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=43).split(X[tv],groups=d.address.iloc[tv]));tr,va=tv[t],tv[v]
assert not set(d.address.iloc[tr]) & set(d.address.iloc[te])
sx=StandardScaler().fit(X[tr]);z=sx.transform(X)
pca=PCA(n_components=2,svd_solver='full').fit(z[tr])
reconstruction=pca.inverse_transform(pca.transform(z))
score=np.mean((z-reconstruction)**2,axis=1);threshold=np.quantile(score[va],.95)
print('Test reconstruction MSE:',score[te].mean(),'Above validation threshold:',np.mean(score[te]>threshold))


## 3. Fit a nonlinear representation and inspect descriptive clusters
The autoencoder fitting routine follows the explicit training/validation pattern in notebook 04. Its bottleneck is two-dimensional, matching PCA dimension but not parameter count.

In [ ]:
from guide_models import fit
model,history,info=fit(X[tr],X[va],task='reconstruction',seed=42)
neural_score=np.mean((z-model.predict(X))**2,axis=1)
print(info,'Test MSE:',neural_score[te].mean())
clusters=KMeans(n_clusters=3,n_init=10,random_state=42).fit(z[tr])
centroids=pd.DataFrame(sx.inverse_transform(clusters.cluster_centers_),columns=FEATURES)
display(centroids)
recorded=pd.read_csv(GUIDE/'metrics.csv');display(recorded[recorded.case.eq('loed')][['model','seed','reconstruction_mse','flag_fraction']])


**Exercise:** can the fraction above a validation threshold be called a false-positive rate? Can cluster 1 be named an attack class?

<details><summary>Feedback after your attempt</summary>

neither is justified without reference labels. A score may reveal capture conditions or unfamiliar behavior. A validation percentile is a review rule, not a distribution-free guarantee under shift. Autoencoder initialization sensitivity is visible in the three archived runs.

</details>

**Adaptation task:** choose an operationally meaningful profile window and audit coverage. Review high-score profiles with independent evidence before assigning fault labels.